# 리딩방 → 건전 투자 유도형 필터링 파이프라인
### 텔레그램 크롤링 → 추천 종목 추출 → 언론 검증 → 기업 건전성 스코어링 → 설명 가능한 규칙 판정 + 통계적 교차검증

원래는 "투자 리딩방 스팸을 이상치로 탐지"하는 프로젝트였고, 이후 "건전한 기업으로 투자금이 흘러가도록 돕는
필터링 시스템"(북극성 B)으로 방향을 넓혔습니다.

**설계 원칙**: 이 도구의 목적은 "이상치 탐지 기법을 쓰는 것"이 아니라 **실제로 투자자를 더 잘 보호하는 것**입니다.
그래서 최종 판단은 왜 경고했는지 사람이 검증할 수 있는 **설명 가능한 규칙**이 내리고, 통계적 이상치 탐지(Isolation
Forest)는 규칙이 놓칠 수 있는 패턴을 추가로 훑어보는 **보조 교차검증 신호**로만 사용합니다. (한때 Isolation
Forest를 최종 판정 엔진으로 세운 버전이 있었지만, 아래 이유로 되돌렸습니다 — 자세한 내용은 맨 아래 참고.)

**전체 흐름**
1. 텔레그램 리딩방 크롤링 (Telethon) — *피처 원천 데이터 수집*
2. 리딩방 메시지에서 추천 종목/기업 추출 — *피처 원천 데이터 수집*
3. 추천 종목 관련 언론 기사 조사, 언론사 신뢰도·뉴스-공시 상관관계 확인 — *피처 엔지니어링*
4. 뉴스 버스트·근접중복 탐지로 "작전세력 의심 보도" 신호 계산 — *피처 엔지니어링*
5. 재무/공시 자료로 기업 안정성·취약도 신호 계산 — *피처 엔지니어링*
6. **설명 가능한 규칙으로 최종 판정** + (비교 종목이 충분할 때만) Isolation Forest로 보조 교차검증
7. 결과 저장 및 다운로드

> **왜 규칙 기반이 메인이고 이상치 탐지는 보조인가?**
> 1. **설명력**: 투자자 보호 도구는 "왜 경고했는지"를 사람이 즉시 검증할 수 있어야 합니다. 규칙은
>    "저신뢰 매체 뉴스 + 공시 없음 + 소형 바이오주 + 거래량 급등"처럼 이유를 코드 그대로 보여주지만,
>    Isolation Forest는 "다른 종목들과 통계적으로 다르다"까지만 말해줄 뿐 방향(좋아서/나빠서)을 모릅니다.
> 2. **표본 크기**: Isolation Forest는 비교할 종목이 충분히 쌓여야 의미가 있습니다. 종목이 몇 개뿐인 지금
>    단계에서 "이상치"를 하나 뽑으라고 하면 사실상 `contamination` 파라미터가 답을 정하는 것과 다르지 않습니다.
> 3. **방향성 오류 방지**: 통계적 이상치는 "달라서 이상한 것"이지 "위험해서 이상한 것"이 아닙니다. 재무구조가
>    유난히 우량한 대형주가 단지 "나머지 종목들과 다르다"는 이유로 위험 신호를 받는 일이 생기면 안 됩니다.
> 그래서 최종 결정은 규칙이 내리고, 통계적 신호는 "규칙도 통과하고 통계적으로도 이상하다"는 이중 확인용으로만
> 덧붙입니다.

> **주의사항**
> - 텔레그램 크롤링은 본인 명의 API 자격증명으로 **공개 채널만** 대상으로 진행하세요.
> - 이 노트북은 기본적으로 `DEMO_MODE = True`로 설정되어 있어, 실제 API 자격증명 없이도
>   내장 샘플 데이터로 전체 파이프라인(1~7단계)을 바로 실행/테스트할 수 있습니다.
> - "작전세력 기사"라는 판별은 통계적 의심 신호일 뿐 법적 판단이 아닙니다. 최종 결론은 반드시 사람이 검토해야 합니다.
> - 3~5단계의 뉴스/기업 재무 데이터는 실제 API 자격증명이 없으면 데모 샘플 데이터로 대체되며, 실제 수치가 아닙니다.

> **회의 반영 (북극성 방향 확정)**: 사기 탐지형(A) vs 건전 투자 유도형(B) 중 **B로 방향을 확정**했습니다.
> 이에 따라 회의에서 지목된 세부 지표 3가지를 규칙에 직접 반영했습니다.
> 1. **언론사 신뢰도 스코어** (3-1단계) — "5만 원이면 네이버 뉴스에 기사를 뿌릴 수 있다"는 지적을 반영해 주요 언론사/소규모 인터넷 언론사를 구분
> 2. **뉴스-주가 상관관계** (3-2단계) — 뉴스 버스트가 실제 공시(실적·계약 등)와 맞물리는지 확인해 "숫자놀음"과 "진짜 호재"를 구분
> 3. **거래량 급등 + 취약도 가중 임계값** (6단계) — 소액으로도 흔들리는 소형주·바이오 기업일수록 더 낮은 의심 점수에서도 경고가 뜨도록 임계값을 동적으로 조정


In [ ]:
!pip install -q telethon datasketch openpyxl scikit-learn finance-datareader

---
## 0단계. 실행 모드 설정

`DEMO_MODE = True`  → 텔레그램/뉴스/재무 API 호출을 모두 건너뛰고 내장 샘플 데이터로 전체 파이프라인을 테스트
`DEMO_MODE = False` → 실제 Telethon 크롤링 + 실제 뉴스 API 호출 수행 (본인 API 자격증명 필요)


In [ ]:
DEMO_MODE = True  # 실제로 크롤링/뉴스 API를 호출하려면 False로 변경하세요


---
# 1단계. 텔레그램 리딩방 크롤링

## 1-1. 수집 (Telethon)

my.telegram.org 에서 발급받은 `api_id`/`api_hash`로 공개 채널 메시지를 수집합니다.


In [ ]:
import json, time, asyncio
from getpass import getpass

RAW_PATH = "telegram_raw.jsonl"

if not DEMO_MODE:
    # Colab/Jupyter 커널은 이미 asyncio 이벤트 루프를 돌리고 있어서,
    # telethon.sync의 동기식 `with client:` 문법은 "You must use async with ..." 에러가 납니다.
    # 따라서 진짜 async/await 문법으로 작성합니다.
    from telethon import TelegramClient

    api_id = int(getpass("api_id: "))
    api_hash = getpass("api_hash: ")
    client = TelegramClient("session_investment_spam", api_id, api_hash)

    async def crawl_channel(channel_username, out_path, limit=3000):
        entity = await client.get_entity(channel_username)
        with open(out_path, "a", encoding="utf-8") as f:
            async for msg in client.iter_messages(entity, limit=limit):
                if not msg.text:
                    continue
                record = {
                    "channel": channel_username,
                    "channel_id": entity.id,
                    "message_id": msg.id,
                    "date": msg.date.isoformat(),
                    "sender_id": msg.sender_id,
                    "text": msg.text,
                    "views": getattr(msg, "views", None),
                    "forwards": getattr(msg, "forwards", None),
                    "fwd_from": str(msg.fwd_from) if msg.fwd_from else None,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        await asyncio.sleep(2)  # rate limit 완화
else:
    print("DEMO_MODE=True → 실제 크롤링은 건너뜁니다. (아래에서 샘플 데이터를 로드합니다)")


In [ ]:
# 실제 크롤링 시 아래 시드 채널 유저네임을 채워서 실행하세요.
SEED_CHANNELS = [
    "gogonero2",
    "aitodaystock",
    "YOUTUBE_INV1",
    "SEDOLSTOCK",
    "CryptoRich_02",
    "CryptoRich33",
    "CryptoRich37",
    "CryptoRich43",
    "coinupkor00",
    "fpt_reviews",
    "mtvpro11",
    "GSoVCrTjCs82ZWE186",
    "SOLOMON_KOR17",
    "daebarkcoincoin"
]

if not DEMO_MODE:
    from telethon.errors import UsernameInvalidError, UsernameNotOccupiedError, FloodWaitError

    failed_channels = []

    async with client:
        for ch in SEED_CHANNELS:
            try:
                await crawl_channel(ch, RAW_PATH)
                print(f"  ✓ {ch} 수집 완료")
            except (UsernameInvalidError, UsernameNotOccupiedError):
                print(f"  ✗ {ch}: 존재하지 않거나 삭제된 채널 (건너뜀)")
                failed_channels.append(ch)
            except FloodWaitError as e:
                print(f"  ✗ {ch}: 요청 제한(FloodWait) {e.seconds}초 대기 필요 — 건너뜀")
                failed_channels.append(ch)
            except Exception as e:
                print(f"  ✗ {ch}: {type(e).__name__} — {e}")
                failed_channels.append(ch)

    print(f"수집 완료 → {RAW_PATH}")
    if failed_channels:
        print(f"실패한 채널 ({len(failed_channels)}개): {failed_channels}")


## 1-2. 채널 확장 (스노우볼 샘플링)

수집된 텍스트에서 `t.me/...` 형태의 채널 링크를 추출해 크롤링 대상을 넓힙니다.


In [ ]:
import re

TME_PATTERN = re.compile(r"t\.me/(joinchat/[\w-]+|\+[\w-]+|[\w_]{5,})")

def extract_new_channels(text):
    return TME_PATTERN.findall(text or "")

if not DEMO_MODE:
    discovered = set()
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            discovered.update(extract_new_channels(rec["text"]))
    new_channels = discovered - set(SEED_CHANNELS)
    print(f"신규 발견 채널 수: {len(new_channels)}")
    # 필요 시 new_channels를 SEED_CHANNELS에 추가해 crawl_channel을 재실행하세요.


## 1-3. 데이터 로드 및 전처리

`DEMO_MODE`에 따라 실제 수집 결과(`telegram_raw.jsonl`) 또는 내장 샘플 데이터를 불러와
정규화 + "추천/시그널 성격" 키워드 매칭을 수행합니다. (도메인 WHOIS·문장 임베딩 등 리딩방 자체의
스팸 여부 판별용 피처는 새 파이프라인의 목적과 맞지 않아 제거했습니다.)


In [ ]:
import pandas as pd

SAMPLE_DATA = [
    # ── 리딩방 스팸으로 의심되는 패턴 (여러 채널에서 유사 문구 반복) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 1, "date": "2026-07-01T09:00:00",
     "sender_id": 5001, "text": "무료 종목 리딩 받아가세요! 이번주 수익률 인증 92% 선착순 카톡 오픈채팅 open.kakao.com/o/gAbC123",
     "views": 15000, "forwards": 320, "fwd_from": None},
    {"channel": "stock_free_2", "channel_id": 1002, "message_id": 1, "date": "2026-07-01T09:05:00",
     "sender_id": 5002, "text": "무료 종목 리딩방 오픈! 이번주 수익 인증 92% 선착순 마감 임박 open.kakao.com/o/gAbC123",
     "views": 14800, "forwards": 305, "fwd_from": None},
    {"channel": "stock_free_3", "channel_id": 1003, "message_id": 1, "date": "2026-07-01T09:10:00",
     "sender_id": 5003, "text": "급등주 무료 리딩 단톡방 초대 수익인증 92% 선착순 open.kakao.com/o/gAbC123",
     "views": 15200, "forwards": 340, "fwd_from": None},
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 2, "date": "2026-07-02T10:00:00",
     "sender_id": 5001, "text": "타점 잡아드립니다 무료 종목 리딩 단톡방 https://t.me/+xYzAbCd",
     "views": 9000, "forwards": 210, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 1, "date": "2026-07-03T11:00:00",
     "sender_id": 5004, "text": "선착순 무료 리딩방 수익 인증 92% open.kakao.com/o/gAbC123 서두르세요",
     "views": 16000, "forwards": 360, "fwd_from": None},
    # ── 실제 종목명을 언급하는 "추천성" 메시지 (2단계 종목 추출 데모용) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 3, "date": "2026-07-05T09:30:00",
     "sender_id": 5001, "text": "현대약품 오늘 상한가 갑니다! 지금 안 사면 후회함, 무료 리딩 신청받아요 open.kakao.com/o/gAbC123",
     "views": 12000, "forwards": 250, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 2, "date": "2026-07-05T10:00:00",
     "sender_id": 5004, "text": "SK하이닉스 급등 시작! 지금 진입 타이밍입니다 단톡방 링크 open.kakao.com/o/gAbC123",
     "views": 13000, "forwards": 270, "fwd_from": None},
    {"channel": "CryptoRich33", "channel_id": 1005, "message_id": 1, "date": "2026-07-06T08:00:00",
     "sender_id": 5005, "text": "이더리움 단기 저점 진입 완료 20% 수익 무료 선물 시그널 신청 https://forms.gle/a8s7wVoQ2DeJiVUv7",
     "views": 8000, "forwards": 150, "fwd_from": None},
    # ── 정상적인 일반 대화/뉴스 공유로 추정되는 메시지 ──
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 1, "date": "2026-07-01T08:00:00",
     "sender_id": 6001, "text": "오늘 코스피 지수는 전일 대비 0.4% 상승 마감했습니다. 외국인 순매수 전환.",
     "views": 500, "forwards": 3, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 2, "date": "2026-07-02T08:00:00",
     "sender_id": 6001, "text": "금일 금통위 기준금리 동결 발표, 시장 예상과 부합.",
     "views": 480, "forwards": 2, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 3, "date": "2026-07-06T08:00:00",
     "sender_id": 6001, "text": "삼성전자 3분기 실적 발표, 시장 예상치에 부합하는 흐름을 보였습니다.",
     "views": 510, "forwards": 4, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 1, "date": "2026-07-02T14:00:00",
     "sender_id": 6002, "text": "재무제표 분석 스터디 이번주 토요일 오후 2시에 진행합니다. 참여 원하시는 분은 댓글 남겨주세요.",
     "views": 120, "forwards": 1, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 2, "date": "2026-07-03T14:00:00",
     "sender_id": 6003, "text": "지난주 스터디 자료 공유드립니다. 링크는 곧 올리겠습니다.",
     "views": 110, "forwards": 0, "fwd_from": None},
    {"channel": "personal_diary_ch", "channel_id": 2003, "message_id": 1, "date": "2026-07-04T20:00:00",
     "sender_id": 6004, "text": "오늘 하루도 고생 많으셨습니다. 내일은 더 좋은 하루가 되길 바랍니다.",
     "views": 40, "forwards": 0, "fwd_from": None},
]

if DEMO_MODE:
    df = pd.DataFrame(SAMPLE_DATA)
else:
    rows = []
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)

# 실제 크롤링 데이터에는 미디어만 있고 텍스트가 없는 메시지(NaN)가 섞여 있을 수 있어 공백으로 채움
df["text"] = df["text"].fillna("")

print(f"로드된 메시지 수: {len(df)}")
df.head()


### 크롤링 원본 데이터 다운로드 (선택)

이후 단계에 들어가기 전에, 지금까지 불러온 원본 메시지 데이터를 먼저 CSV/XLSX로
저장하고 다운로드하고 싶다면 아래 셀을 실행하세요.


In [ ]:
RAW_CSV_PATH = "telegram_raw_messages.csv"
RAW_XLSX_PATH = "telegram_raw_messages.xlsx"

df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")
df.to_excel(RAW_XLSX_PATH, index=False)
print(f"저장 완료 → {RAW_CSV_PATH}, {RAW_XLSX_PATH} ({len(df)}건)")

try:
    from google.colab import files
    files.download(RAW_CSV_PATH)
    files.download(RAW_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


In [ ]:
# 내부 FDS 가이드라인 3.1 검색어 사전을 그대로 카테고리화 (추천/과장/모집/위험/정상비교)
KEYWORD_CATEGORIES = {
    "추천": [r"추천주", r"오늘의\s*종목", r"매수가", r"목표가", r"손절가", r"익절",
           r"시초가\s*매매", r"종가\s*매매", r"단타", r"포착\s*종목"],
    "과장": [r"상한가", r"급등\s*예상주", r"세력주", r"주포", r"재료\s*공개", r"무조건", r"확실", r"마지막\s*기회"],
    "모집": [r"무료\s*(리딩|종목)", r"VIP\s*방", r"본방\s*입장", r"무료\s*체험", r"수익\s*인증",
           r"승률", r"잔고\s*인증", r"선착순", r"멤버(?:십|쉽)", r"단톡방", r"카톡\s*오픈채팅"],
    "위험": [r"원금\s*보장", r"수익\s*보장", r"손실\s*보전", r"대리매매", r"입금", r"거래소\s*가입"],
    "정상비교": [r"장전\s*브리핑", r"증시\s*일정", r"기업\s*공시", r"산업\s*리포트", r"수급\s*동향", r"실적\s*발표"],
}

def normalize(text):
    text = re.sub(r"[\u200b\uFE0F\u3164]", "", text)  # 제로폭/변형 문자 제거
    return text.strip()

def category_hits(text, category):
    return sum(bool(re.search(p, text or "")) for p in KEYWORD_CATEGORIES[category])

df["text_norm"] = df["text"].map(normalize)
for _category in KEYWORD_CATEGORIES:
    df[f"kw_{_category}"] = df["text_norm"].map(lambda t, c=_category: category_hits(t, c))

df["kw_hits"] = df[[f"kw_{c}" for c in KEYWORD_CATEGORIES]].sum(axis=1)  # 기존 코드 호환용 총합

df[["channel", "text_norm"] + [f"kw_{c}" for c in KEYWORD_CATEGORIES]]

### 참고: 채널 유형 프로파일링 (내부 FDS 가이드라인 2장)

가이드라인은 "리딩형이라는 이유만으로 사기 채널로 분류하지 않는다"고 명시합니다. 그래서 채널을 곧바로
위험/정상으로 나누지 않고, 우선 **A 정상비교군 / B 리딩형 / C 광고·모집형 / D 이상행위 후보군**으로
프로파일링만 해둡니다. 이 라벨은 참고용 설명 자료이며, 최종 `investment_guidance` 판정에는 아직 반영하지
않습니다(판정 로직은 6단계의 설명 가능한 규칙이 그대로 유지).


In [ ]:
channel_profile = df.groupby("channel")[[f"kw_{c}" for c in KEYWORD_CATEGORIES]].sum()

def classify_channel_type(row):
    # 가이드라인 2장 우선순위: 정상비교군 신호가 있고 모집/위험 신호가 없으면 A, 위험 신호가 있으면 D,
    # 모집 신호가 있으면 C, 그 외 추천/과장 신호만 있으면 B로 프로파일링
    if row["kw_정상비교"] > 0 and row["kw_모집"] == 0 and row["kw_위험"] == 0:
        return "A_정상비교군"
    if row["kw_위험"] > 0:
        return "D_이상행위후보군"
    if row["kw_모집"] > 0:
        return "C_광고모집형"
    if row["kw_추천"] > 0 or row["kw_과장"] > 0:
        return "B_리딩형"
    return "미분류"

channel_profile["channel_type"] = channel_profile.apply(classify_channel_type, axis=1)
channel_profile

---
# 2단계. 리딩방 메시지에서 추천 종목/기업 추출

`FinanceDataReader`로 KRX(코스피+코스닥) 상장 종목 전체 리스트를 받아와 사전을 구성합니다. 네트워크로 목록을
못 가져오면(오프라인 환경 등) 데모용 소수 종목으로 자동 대체합니다. 사전 매칭은 여전히 완벽한 개체명 인식(NER)은
아니므로, 아래 셀에서 짧은 종목명이 긴 종목명의 일부를 잘못 채가지 않도록 간단한 보정도 함께 넣었습니다.


In [ ]:
KRX_LISTING_DF = None  # 5단계에서 시가총액·업종을 재사용할 수 있도록 전역에 보관

def load_krx_stock_dictionary():
    global KRX_LISTING_DF
    import FinanceDataReader as fdr
    krx = fdr.StockListing("KRX")  # 코스피+코스닥+코넥스 전 종목, 컬럼: Code, Name, Market, Marcap, Sector 등
    krx = krx.dropna(subset=["Code", "Name"])
    KRX_LISTING_DF = krx
    return [{"name": n, "ticker": c, "asset_type": "주식"} for n, c in zip(krx["Name"], krx["Code"])]

CRYPTO_DICTIONARY = [
    {"name": "이더리움", "ticker": None, "asset_type": "코인"},
    {"name": "비트코인", "ticker": None, "asset_type": "코인"},
]

DEMO_STOCK_DICTIONARY = [
    {"name": "현대약품", "ticker": "004310", "asset_type": "주식"},
    {"name": "SK하이닉스", "ticker": "000660", "asset_type": "주식"},
    {"name": "삼성전자", "ticker": "005930", "asset_type": "주식"},
]

try:
    STOCK_DICTIONARY = load_krx_stock_dictionary() + CRYPTO_DICTIONARY
    print(f"KRX 상장 종목 {len(STOCK_DICTIONARY) - len(CRYPTO_DICTIONARY)}개 + 코인 {len(CRYPTO_DICTIONARY)}개 사전 로드 완료")
except Exception as e:
    print(f"KRX 종목 리스트 로드 실패({type(e).__name__}: {e}) → 데모용 소수 종목 사전으로 대체합니다.")
    STOCK_DICTIONARY = DEMO_STOCK_DICTIONARY + CRYPTO_DICTIONARY

# 짧은 종목명이 긴 종목명의 일부를 먼저 가로채지 않도록 긴 이름부터 확인
STOCK_DICTIONARY = sorted(STOCK_DICTIONARY, key=lambda item: len(item["name"]), reverse=True)

def extract_recommended_stocks(text):
    text = text or ""
    candidates = [item["name"] for item in STOCK_DICTIONARY if item["name"] in text]
    # 예: "SK하이닉스"가 매칭되면, 그 안에 포함된 "SK" 같은 짧은 매칭은 제거
    return [name for name in candidates if not any(name != other and name in other for other in candidates)]

df["recommended_stocks"] = df["text_norm"].map(extract_recommended_stocks)

# 메시지 단위 → (채널, 메시지, 종목) 단위로 펼치기
mention_rows = []
for _, row in df.iterrows():
    for stock in row["recommended_stocks"]:
        mention_rows.append({
            "channel": row["channel"],
            "message_id": row["message_id"],
            "date": row["date"],
            "stock_name": stock,
            "kw_hits": row["kw_hits"],
            "text_norm": row["text_norm"],
        })

mentions_df = pd.DataFrame(mention_rows)
print(f"추출된 종목 언급 수: {len(mentions_df)}건 (메시지 {len(df)}건 중)")

if len(mentions_df):
    mention_summary = (
        mentions_df.groupby("stock_name")
        .agg(mention_count=("message_id", "count"), channel_count=("channel", "nunique"),
             first_mentioned=("date", "min"))
        .reset_index()
        .sort_values("mention_count", ascending=False)
    )
else:
    mention_summary = pd.DataFrame(columns=["stock_name", "mention_count", "channel_count", "first_mentioned"])

mention_summary

---
# 3단계. 추천 종목 관련 언론 기사 조사

네이버 뉴스 검색 API(`https://openapi.naver.com/v1/search/news.json`)로 각 추천 종목의 최근 기사를 조회합니다.
Client ID/Secret은 [네이버 개발자센터](https://developers.naver.com/apps/#/register)에서 애플리케이션 등록 후 발급받습니다.


In [ ]:
import requests
from urllib.parse import quote

SAMPLE_NEWS = [
    # ── "현대약품" 관련 — 동시다발적 보도자료성 기사(작전 의심 패턴) ──
    {"stock_name": "현대약품", "title": "현대약품, 특징주 부각...단기 급등세",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이고 있다. 매수 문의가 폭주하는 모습이다.",
     "pubDate": "2026-07-05T11:00:00", "link": "https://press-release-a.example.com/1"},
    {"stock_name": "현대약품", "title": "현대약품 특징주 부각, 단기 급등세 지속",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 지속하고 있다. 매수 문의가 폭주하는 상황.",
     "pubDate": "2026-07-05T11:30:00", "link": "https://press-release-b.example.com/1"},
    {"stock_name": "현대약품", "title": "[특징주] 현대약품, 급등세... 매수 문의 폭주",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이는 중이다. 매수 문의가 폭주하고 있다.",
     "pubDate": "2026-07-05T12:00:00", "link": "https://press-release-c.example.com/1"},
    # ── "SK하이닉스" 관련 — 정상적인 실적/업황 기사 (한국경제 = KPF 참여매체) ──
    {"stock_name": "SK하이닉스", "title": "SK하이닉스, HBM 수요 확대에 3분기 실적 개선 전망",
     "description": "증권가는 SK하이닉스의 HBM 수요 확대로 3분기 실적이 시장 예상치를 상회할 것으로 내다봤다.",
     "pubDate": "2026-07-05T09:00:00", "link": "https://www.hankyung.com/article/2026070500001"},
    # ── "삼성전자" 관련 — 정상적인 실적 기사 (매일경제 = KPF 참여매체) ──
    {"stock_name": "삼성전자", "title": "삼성전자 3분기 실적 예상치 부합, 반도체 업황 개선세",
     "description": "삼성전자가 3분기 실적을 발표하며 시장 예상치에 부합하는 흐름을 보였다.",
     "pubDate": "2026-07-06T09:00:00", "link": "https://www.mk.co.kr/article/2026070600001"},
]

def fetch_naver_news(query, display=20, client_id=None, client_secret=None):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {"X-Naver-Client-Id": client_id, "X-Naver-Client-Secret": client_secret}
    params = {"query": query, "display": display, "sort": "date"}
    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return [
        {
            "stock_name": query,
            "title": re.sub("<.*?>", "", it["title"]),
            "description": re.sub("<.*?>", "", it["description"]),
            "pubDate": it["pubDate"],
            "link": it["link"],
        }
        for it in items
    ]

if not DEMO_MODE:
    naver_client_id = getpass("네이버 뉴스 API Client ID: ")
    naver_client_secret = getpass("네이버 뉴스 API Client Secret: ")

    news_rows = []
    for stock in mention_summary["stock_name"]:
        try:
            news_rows.extend(fetch_naver_news(stock, client_id=naver_client_id, client_secret=naver_client_secret))
        except Exception as e:
            print(f"  ✗ {stock} 뉴스 조회 실패: {type(e).__name__} — {e}")
    news_df = pd.DataFrame(news_rows)
else:
    print("DEMO_MODE=True → 실제 뉴스 API 호출을 건너뛰고 샘플 기사 데이터를 사용합니다.")
    news_df = pd.DataFrame(SAMPLE_NEWS)

news_df["press_domain"] = news_df["link"].map(lambda l: re.search(r"https?://([\w.-]+)", l).group(1) if l else None)
print(f"수집된 기사 수: {len(news_df)}건")
news_df


## 3-1. 언론사 신뢰도 분류
### (회의 반영: 언론사 신뢰도 스코어 — KPF 참여 언론사 + 뉴스 디렉터리 도메인 매핑)

회의에서 나온 핵심 지적 — "리딩방에서 5만 원이면 네이버 뉴스에 기사를 뿌릴 수 있다" — 을 반영해,
**한국언론진흥재단(KPF)이 공식 발표한 "참여 언론사 현황"**을 1차 화이트리스트로 쓰고, 뉴스 링크 모음 사이트
(sinmundeul.com)에서 확인한 실제 도메인으로 매체명-도메인 매핑을 보강했습니다.

또한 **연합뉴스·JTBC·뉴시스처럼 실제로는 신뢰도 높은 매체가 KPF의 이 특정 참여사 명단에는 없는 경우**가 있어,
이런 "명단 밖이지만 잘 알려진 주요 매체"는 별도로 `OTHER_MAJOR_MEDIA`에 등록해 중간 신뢰도를 부여합니다.
즉 신뢰도는 3단계입니다: **① KPF 참여매체(카테고리별 차등) > ② 기타 알려진 주요 매체 > ③ 명단에 없는 매체(미검증)**.


In [ ]:
# 한국언론진흥재단(KPF) "참여 언론사 현황" (2025년 9월 기준, 120개 언론사 131개 매체)
# 출처: https://www.kpf.or.kr/front/intropage/intropageShow.do?page_id=08cccf3f3cf549d29f97c04304cab50c
KPF_PARTICIPATING_MEDIA = {
    "전국종합일간": ["경향신문", "국민일보", "내일신문", "동아일보", "문화일보", "서울신문", "세계일보",
                 "아시아투데이", "조선일보", "중앙일보", "한겨레", "한국일보"],
    "지역종합일간": ["강원도민일보", "강원일보", "경기신문", "경기일보", "경남도민일보", "경남신문", "경남일보",
                 "경북도민일보", "경북매일신문", "경북일보", "경상일보", "경인일보", "광남일보", "광주매일신문",
                 "광주일보", "국제신문", "금강일보", "기호일보", "남도일보", "대구신문", "대구일보", "대전일보",
                 "동양일보", "매일신문", "무등일보", "부산일보", "새전북신문", "영남일보", "울산매일", "울산신문",
                 "인천일보", "전남일보", "전라일보", "전북도민일보", "전북일보", "제민일보", "제주일보", "중도일보",
                 "중부매일", "중부일보", "충북일보", "충청일보", "충청타임즈", "충청투데이", "한라일보"],
    "경제일간": ["e대한경제", "매일경제", "머니투데이", "메트로경제", "브릿지경제", "서울경제", "아시아경제",
              "아주경제", "에너지경제", "이데일리", "이투데이", "파이낸셜뉴스", "한국경제", "헤럴드경제"],
    "스포츠일간": ["스포츠경향", "스포츠동아", "스포츠서울", "스포츠월드", "일간스포츠"],
    "영자일간": ["코리아중앙데일리", "코리아타임스", "코리아헤럴드"],
    "전문일간·어린이신문": ["농민신문", "디지털타임스", "소년한국일보", "어린이동아", "전자신문", "환경일보"],
    "종합·전문주간": ["기자협회보", "미디어오늘", "시사IN", "이코노미스트", "일요신문", "주간한국",
                  "중앙선데이", "투데이신문", "한겨레21"],
    "지역주간": ["고양신문", "광양신문", "김포신문", "뉴스서천", "당진시대", "영암우리신문", "영주시민신문",
              "옥천신문", "원주투데이", "주간설악신문", "평택시민신문", "홍성신문"],
    "인터넷신문": ["EBN", "PD저널", "노컷뉴스", "뉴스펭귄", "뉴스핌", "데일리안", "미디어펜", "브레이크뉴스",
              "비즈워치", "서울와이어", "스포츠한국", "어린이강원일보", "여성경제신문", "이코노믹데일리",
              "지디넷코리아", "쿠키뉴스", "펜앤드마이크", "프레시안", "헬로디디"],
    "방송사": ["KBS", "MBC", "MBN", "OBS", "SBS", "YTN"],
}

# 카테고리별 신뢰도 가중치 — 전국 단위 일간지·방송사를 가장 높게, 주간지·지역지로 갈수록 낮게 설정한
# 편집상 도달 범위 기준 초기값입니다(공식 검증된 가중치는 아니므로 실제 사례로 재조정 필요).
CATEGORY_CREDIBILITY = {
    "전국종합일간": 1.0,
    "방송사": 1.0,
    "경제일간": 0.9,
    "영자일간": 0.8,
    "스포츠일간": 0.8,
    "전문일간·어린이신문": 0.7,
    "지역종합일간": 0.7,
    "인터넷신문": 0.6,
    "종합·전문주간": 0.6,
    "지역주간": 0.5,
}

MEDIA_NAME_TO_CATEGORY = {
    name: category
    for category, names in KPF_PARTICIPATING_MEDIA.items()
    for name in names
}

# 매체명 -> 실제 사용되는 도메인(들). 뉴스 서브도메인을 따로 쓰는 방송사 등은 여러 도메인을 등록합니다.
# 출처: sinmundeul.com(뉴스 링크 모음 사이트)에서 확인한 실제 링크 + 직접 확인한 주요 매체.
# 모르는 도메인은 추측해서 넣지 않았고, 여전히 비어 있는 매체(주로 소규모 지역주간지)는 TODO로 남깁니다.
MEDIA_NAME_TO_DOMAINS = {
    "경향신문": ["khan.co.kr"], "국민일보": ["kmib.co.kr"], "내일신문": ["naeil.com"],
    "동아일보": ["donga.com"], "문화일보": ["munhwa.com"], "서울신문": ["seoul.co.kr"],
    "세계일보": ["segye.com"], "조선일보": ["chosun.com"], "중앙일보": ["joongang.co.kr"],
    "한겨레": ["hani.co.kr"], "한국일보": ["hankookilbo.com"],
    "매일경제": ["mk.co.kr"], "머니투데이": ["mt.co.kr", "moneytoday.co.kr"], "서울경제": ["sedaily.com"],
    "아시아경제": ["asiae.co.kr"], "이데일리": ["edaily.co.kr"], "파이낸셜뉴스": ["fnnews.com"],
    "한국경제": ["hankyung.com"], "헤럴드경제": ["heraldcorp.com", "biz.heraldcorp.com"],
    "뉴스핌": ["newspim.com"],
    "전자신문": ["etnews.com"], "디지털타임스": ["dt.co.kr"],
    "코리아헤럴드": ["koreaherald.com"], "코리아타임스": ["koreatimes.co.kr"],
    "KBS": ["kbs.co.kr", "news.kbs.co.kr"], "MBC": ["imbc.com", "imnews.imbc.com"],
    "SBS": ["sbs.co.kr", "news.sbs.co.kr"], "YTN": ["ytn.co.kr"], "MBN": ["mbn.co.kr"],
    "OBS": ["obs.co.kr"],
    "프레시안": ["pressian.com"], "노컷뉴스": ["nocutnews.co.kr"], "데일리안": ["dailian.co.kr"],
    "미디어펜": ["mediapen.com"], "지디넷코리아": ["zdnet.co.kr"],
    "시사IN": ["sisain.co.kr"], "미디어오늘": ["mediatoday.co.kr"],
    "한겨레21": ["h21.hani.co.kr"],
    "스포츠동아": ["sports.donga.com"], "스포츠서울": ["sportsseoul.com"],
    "스포츠경향": ["sports.khan.co.kr"], "스포츠월드": ["sportsworldi.com"], "일간스포츠": ["isplus.com"],
    "부산일보": ["busan.com"], "국제신문": ["kookje.co.kr"], "영남일보": ["yeongnam.com"],
    "매일신문": ["imaeil.com"], "경인일보": ["kyeongin.com"], "중부일보": ["joongboo.com"],
    "대전일보": ["daejonilbo.com"], "충청투데이": ["cctoday.co.kr"], "광주일보": ["kwangju.co.kr"],
    "전남일보": ["jnilbo.com"], "전북일보": ["jjan.kr"], "강원일보": ["kwnews.co.kr"],
    "제주일보": ["jeju.co.kr"], "제민일보": ["jemin.com"], "경남신문": ["knnews.co.kr"],
}
DOMAIN_TO_MEDIA_NAME = {
    domain: name
    for name, domains in MEDIA_NAME_TO_DOMAINS.items()
    for domain in domains
}

# KPF의 "참여 언론사 현황"은 KPF 프로그램 참여사 목록일 뿐이라, 연합뉴스·JTBC처럼 실제로는 신뢰도 높은
# 매체가 이 특정 명단에는 없을 수 있습니다. 그런 "명단 밖이지만 잘 알려진 주요 매체"는 여기 따로 등록해
# 중간 수준의 신뢰도를 부여합니다(출처: sinmundeul.com 뉴스 디렉터리 + 상식적으로 알려진 매체).
OTHER_MAJOR_MEDIA = {
    "연합뉴스": ["yonhapnews.co.kr"], "뉴시스": ["newsis.com"],
    "JTBC": ["jtbc.co.kr"], "TV조선": ["tvchosun.com"], "채널A": ["ichannela.com"],
    "연합뉴스TV": ["yonhapnewstv.co.kr"],
    "오마이뉴스": ["ohmynews.com"], "뉴스타파": ["newstapa.org"],
    "조선비즈": ["biz.chosun.com"], "더벨": ["thebell.co.kr"], "비즈니스포스트": ["businesspost.co.kr"],
    "한국경제TV": ["wowtv.co.kr"], "스포츠조선": ["sports.chosun.com"], "MK스포츠": ["sports.mk.co.kr"],
    "시사저널": ["sisajournal.com"],
    "주간조선": ["weekly.chosun.com"], "주간동아": ["weekly.donga.com"],
    "월간조선": ["monthly.chosun.com"], "신동아": ["shindonga.donga.com"],
}
OTHER_MAJOR_DOMAIN_TO_NAME = {
    domain: name
    for name, domains in OTHER_MAJOR_MEDIA.items()
    for domain in domains
}
OTHER_MAJOR_CREDIBILITY = 0.85

# KPF 명단에도, 기타 주요 매체 명단에도 없는(또는 아직 도메인 매핑이 없는) 매체는 "검증되지 않은 매체"로
# 보수적으로 취급
DEFAULT_PRESS_TIER = {"tier": "명단 외(미검증)", "credibility": 0.3}

def classify_press(domain):
    if domain and domain.startswith("www."):
        domain = domain[len("www."):]

    media_name = DOMAIN_TO_MEDIA_NAME.get(domain)
    if media_name:
        category = MEDIA_NAME_TO_CATEGORY[media_name]
        return {"tier": f"KPF 참여매체 · {category}", "credibility": CATEGORY_CREDIBILITY[category]}

    other_name = OTHER_MAJOR_DOMAIN_TO_NAME.get(domain)
    if other_name:
        return {"tier": "기타 주요 매체(KPF 명단 외)", "credibility": OTHER_MAJOR_CREDIBILITY}

    return DEFAULT_PRESS_TIER

news_df["press_tier"] = news_df["press_domain"].map(lambda d: classify_press(d)["tier"])
news_df["press_credibility"] = news_df["press_domain"].map(lambda d: classify_press(d)["credibility"])

news_df[["stock_name", "press_domain", "press_tier", "press_credibility"]]

## 3-2. 뉴스-주가 상관관계 확인
### (회의 반영: 뉴스-주가 상관관계 — Open DART 실제 연동)

뉴스가 몰렸다고 해서 전부 의심스러운 건 아닙니다. **실제 공시(실적·계약·임상 등)와 맞물려 있다면 정상적인 호재**이고,
공시 없이 뉴스만으로 가격·거래량이 흔들렸다면 세력의 "숫자놀음"에 가깝습니다.

[Open DART](https://opendart.fss.or.kr/guide/main.do?apiGrpCd=DS001)에서 무료 인증키를 발급받아
① 전체 상장사의 종목코드↔고유번호 매핑(`corpCode.xml`)을 받고, ② 공시검색 API(`list.json`)로 리딩방
추천 시점 기준 **전 7일 · 당일 · 후 3일** 구간에 실제 공시가 있었는지 조회합니다(내부 FDS 가이드라인 8.2절).


In [ ]:
import zipfile, io
import xml.etree.ElementTree as ET

DART_CORP_CODE_URL = "https://opendart.fss.or.kr/api/corpCode.xml"
DART_LIST_URL = "https://opendart.fss.or.kr/api/list.json"

# 가이드라인 8.2: 추천 시점 기준 전 7일 · 당일 · 후 3일 구간의 공시를 조회
DISCLOSURE_WINDOW_BEFORE_DAYS = 7
DISCLOSURE_WINDOW_AFTER_DAYS = 3

def load_dart_corp_codes(api_key):
    resp = requests.get(DART_CORP_CODE_URL, params={"crtfc_key": api_key}, timeout=15)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_bytes = zf.read(zf.namelist()[0])
    root = ET.fromstring(xml_bytes)
    rows = [
        {
            "corp_code": corp.findtext("corp_code"),
            "corp_name": corp.findtext("corp_name"),
            "stock_code": (corp.findtext("stock_code") or "").strip(),
        }
        for corp in root.findall("list")
    ]
    return pd.DataFrame([r for r in rows if r["stock_code"]])  # 비상장 법인(종목코드 없음)은 제외

def fetch_dart_disclosures(corp_code, bgn_de, end_de, api_key):
    params = {
        "crtfc_key": api_key, "corp_code": corp_code,
        "bgn_de": bgn_de, "end_de": end_de,
        "page_no": 1, "page_count": 100,
    }
    resp = requests.get(DART_LIST_URL, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return data.get("list", []) if data.get("status") == "000" else []

if not DEMO_MODE:
    dart_api_key = getpass("Open DART 인증키: ")
    corp_code_df = load_dart_corp_codes(dart_api_key)
    ticker_to_corp_code = corp_code_df.set_index("stock_code")["corp_code"].to_dict()

    # 우선주(예: "대덕1우", "두산2우B")는 DART 관점에서 별도 법인이 아니라 보통주를 발행한 법인과 동일한
    # 법인입니다. DART corpCode.xml은 그 법인의 대표 종목코드(보통주)로만 등록돼 있어, 우선주 티커로는
    # 절대 매핑되지 않습니다. 이름에서 우선주 표기를 떼어 보통주 이름으로 되돌린 뒤 재조회합니다.
    PREFERRED_STOCK_SUFFIX_PATTERN = re.compile(r"\d*우[A-Z]?$")

    def find_common_stock_ticker(stock_name):
        base_name = PREFERRED_STOCK_SUFFIX_PATTERN.sub("", stock_name)
        if base_name == stock_name:
            return None  # 애초에 우선주 표기가 아니었음
        match = next((item for item in STOCK_DICTIONARY if item["name"] == base_name), None)
        return match["ticker"] if match else None

    def resolve_corp_code(stock_name, ticker):
        if ticker and ticker in ticker_to_corp_code:
            return ticker_to_corp_code[ticker]
        common_ticker = find_common_stock_ticker(stock_name)
        if common_ticker and common_ticker in ticker_to_corp_code:
            return ticker_to_corp_code[common_ticker]
        return None

    disclosure_rows = []
    for _, row in mention_summary.iterrows():
        stock_name = row["stock_name"]
        stock_info = next((item for item in STOCK_DICTIONARY if item["name"] == stock_name), None)
        ticker = stock_info["ticker"] if stock_info else None
        corp_code = resolve_corp_code(stock_name, ticker)

        has_disclosure = False
        if corp_code:
            mention_date = pd.to_datetime(row["first_mentioned"])
            bgn_de = (mention_date - pd.Timedelta(days=DISCLOSURE_WINDOW_BEFORE_DAYS)).strftime("%Y%m%d")
            end_de = (mention_date + pd.Timedelta(days=DISCLOSURE_WINDOW_AFTER_DAYS)).strftime("%Y%m%d")
            try:
                disclosures = fetch_dart_disclosures(corp_code, bgn_de, end_de, dart_api_key)
                has_disclosure = len(disclosures) > 0
            except Exception as e:
                print(f"  ✗ {stock_name} 공시 조회 실패: {type(e).__name__} — {e}")
        else:
            print(f"  · {stock_name}: DART 법인 매핑 없음(코인이거나 비상장 등) → 공시 없음으로 처리")

        disclosure_rows.append({"stock_name": stock_name, "has_official_disclosure": has_disclosure})

    disclosure_check = pd.DataFrame(disclosure_rows)
else:
    print("DEMO_MODE=True → 실제 DART 조회를 건너뛰고 샘플 공시 데이터를 사용합니다.")
    SAMPLE_DISCLOSURES = {
        "현대약품": False,   # 뉴스 버스트 기간에 매칭되는 공식 공시 없음 → 뉴스만으로 만든 움직임 의심
        "SK하이닉스": True,  # 실적 관련 공시 존재
        "삼성전자": True,
    }
    disclosure_check = pd.DataFrame({
        "stock_name": list(SAMPLE_DISCLOSURES.keys()),
        "has_official_disclosure": list(SAMPLE_DISCLOSURES.values()),
    })

disclosure_check

---
# 4단계. 뉴스 버스트·근접중복 탐지로 "작전세력 의심 보도" 1차 판별

정상적인 기업 뉴스라면 매체마다 문구가 다르고 실적·공시 같은 사실관계를 전달합니다. 반면 리딩방發 펌핑을
뒷받침하려는 보도자료성 기사는 **여러 매체가 짧은 기간에 거의 동일한 문구를 반복 게시**하는 경향이 있습니다.
이 "근접중복 반복성"은 원래 리딩방 스팸 탐지에 쓰던 MinHash 기법을 그대로 재사용해 잡아냅니다.
여기서 계산한 `pump_news_score`·`news_price_mismatch`는 6단계에서 설명 가능한 규칙의 입력으로 쓰입니다.


In [ ]:
from datasketch import MinHash, MinHashLSH

PUMP_PR_KEYWORDS = [r"특징주", r"급등세", r"매수\s*문의\s*폭주", r"단기\s*급등", r"테마\s*부각"]

def get_minhash(text, num_perm=64, shingle_size=4):
    # 한국어는 조사·어미가 붙어 공백 기준 단어 분리가 불안정하므로,
    # 공백을 제거한 문자 단위 n-gram(shingle)으로 유사도를 비교합니다.
    text = "".join(text.split())
    m = MinHash(num_perm=num_perm)
    shingles = {text[i:i + shingle_size] for i in range(max(len(text) - shingle_size + 1, 1))}
    for sh in shingles:
        m.update(sh.encode("utf8"))
    return m

def pr_keyword_hits(text):
    return sum(bool(re.search(p, text or "")) for p in PUMP_PR_KEYWORDS)

news_df["combined_text"] = (news_df["title"].fillna("") + " " + news_df["description"].fillna(""))
news_df["pr_keyword_hits"] = news_df["combined_text"].map(pr_keyword_hits)

lsh = MinHashLSH(threshold=0.3, num_perm=64)
duplicate_count = []
for idx, text in enumerate(news_df["combined_text"]):
    mh = get_minhash(text)
    matches = lsh.query(mh)
    duplicate_count.append(len(matches))
    lsh.insert(str(idx), mh)
news_df["duplicate_count"] = duplicate_count

# 리딩방에서 해당 종목이 처음 언급된 시점 대비, N일 이내 몰린 기사 수(버스트) 계산
BURST_WINDOW_DAYS = 3
first_mention = mention_summary.set_index("stock_name")["first_mentioned"] if len(mention_summary) else pd.Series(dtype="object")

def days_since_mention(row):
    if row["stock_name"] not in first_mention.index:
        return None
    mention_date = pd.to_datetime(first_mention[row["stock_name"]])
    news_date = pd.to_datetime(row["pubDate"])
    return (news_date - mention_date).total_seconds() / 86400

news_df["days_since_mention"] = news_df.apply(days_since_mention, axis=1)
news_df["is_burst"] = news_df["days_since_mention"].between(0, BURST_WINDOW_DAYS)

def normalize01(series):
    s = series.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng else s * 0

stock_news = news_df.groupby("stock_name").agg(
    article_count=("title", "count"),
    avg_duplicate_count=("duplicate_count", "mean"),
    total_pr_keyword_hits=("pr_keyword_hits", "sum"),
    burst_count=("is_burst", "sum"),
    minor_press_ratio=("press_credibility", lambda s: (s < 0.5).mean()),
).reset_index()

# 회의 반영: 공시 없이 뉴스만으로 몰린 경우("뉴스-주가 상관관계" 미스매치)를 별도 신호로 포함
stock_news = stock_news.merge(disclosure_check, on="stock_name", how="left")
stock_news["has_official_disclosure"] = stock_news["has_official_disclosure"].fillna(False)
stock_news["news_price_mismatch"] = (stock_news["burst_count"] > 0) & (~stock_news["has_official_disclosure"])

stock_news["pump_news_score"] = (
    0.25 * normalize01(stock_news["avg_duplicate_count"])
    + 0.15 * normalize01(stock_news["total_pr_keyword_hits"])
    + 0.2 * normalize01(stock_news["burst_count"])
    + 0.25 * stock_news["minor_press_ratio"]
    + 0.15 * stock_news["news_price_mismatch"].astype(float)
)

stock_news.sort_values("pump_news_score", ascending=False)[[
    "stock_name", "article_count", "avg_duplicate_count", "total_pr_keyword_hits",
    "burst_count", "minor_press_ratio", "has_official_disclosure", "news_price_mismatch", "pump_news_score",
]]


---
# 5단계. 기업 건전성 스코어링

재무/공시 지표로 "이 회사가 안정적인 회사인지"를 스코어링합니다. Open DART로 부채비율(재무제표 주요계정)·
감사의견·관리종목 지정 이력(공시 목록에서 검색)·최대주주 지분율 변동을 조회하고, 시가총액·업종은 2단계에서
이미 받아온 KRX 상장 종목 리스트(`KRX_LISTING_DF`)를 재사용합니다.


In [ ]:
DART_FNLTT_URL = "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json"
DART_AUDIT_OPINION_URL = "https://opendart.fss.or.kr/api/accnutAdtorNmNdAdtOpinion.json"
DART_MAJOR_HOLDER_CHANGE_URL = "https://opendart.fss.or.kr/api/hyslrChgSttus.json"

DEFAULT_REPRT_CODE = "11011"  # 사업보고서(연간) — 반기/분기가 필요하면 11012/11013/11014로 교체
DEFAULT_BSNS_YEAR = str(pd.Timestamp.now().year - 1)  # 가장 최근 확정된 사업연도

def fetch_dart_json(url, params):
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return data.get("list", []) if data.get("status") == "000" else []

def fetch_debt_ratio(corp_code, api_key, bsns_year=DEFAULT_BSNS_YEAR, reprt_code=DEFAULT_REPRT_CODE):
    rows = fetch_dart_json(DART_FNLTT_URL, {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code": reprt_code,
    })
    accounts = {r.get("account_nm"): r.get("thstrm_amount") for r in rows}
    debt, equity = accounts.get("부채총계"), accounts.get("자본총계")
    if debt is None or equity is None:
        return None
    debt, equity = float(str(debt).replace(",", "")), float(str(equity).replace(",", ""))
    return (debt / equity * 100) if equity else None

def fetch_audit_opinion(corp_code, api_key, bsns_year=DEFAULT_BSNS_YEAR, reprt_code=DEFAULT_REPRT_CODE):
    rows = fetch_dart_json(DART_AUDIT_OPINION_URL, {
        "crtfc_key": api_key, "corp_code": corp_code, "bsns_year": bsns_year, "reprt_code": reprt_code,
    })
    return rows[0].get("adt_opinion") if rows else None  # 예: "적정"

def fetch_major_holder_change_pct(corp_code, api_key):
    rows = fetch_dart_json(DART_MAJOR_HOLDER_CHANGE_URL, {"crtfc_key": api_key, "corp_code": corp_code})
    if not rows:
        return 0.0
    latest = rows[0]
    try:
        before = float(str(latest.get("bfr_hold_pct", "0") or "0").replace(",", ""))
        after = float(str(latest.get("trmend_hold_pct", "0") or "0").replace(",", ""))
        return after - before
    except (TypeError, ValueError):
        return 0.0

def fetch_administrative_issue(corp_code, api_key, lookback_days=730):
    end_de = pd.Timestamp.now().strftime("%Y%m%d")
    bgn_de = (pd.Timestamp.now() - pd.Timedelta(days=lookback_days)).strftime("%Y%m%d")
    disclosures = fetch_dart_disclosures(corp_code, bgn_de, end_de, api_key)  # 3-2단계에서 정의됨
    return int(any("관리종목" in (d.get("report_nm") or "") for d in disclosures))

def lookup_krx_field(ticker, column):
    if KRX_LISTING_DF is None or ticker is None:
        return None
    row = KRX_LISTING_DF[KRX_LISTING_DF["Code"] == ticker]
    if row.empty or column not in row.columns:
        return None
    return row.iloc[0][column]

def guess_sector_label(ticker):
    # 실제 업종분류가 필요하면 KRX 업종코드 매핑으로 교체해야 합니다. 여기서는 KRX 리스팅의 업종 텍스트에서
    # "바이오/제약/의약품" 키워드가 있는지로 근사치만 추정합니다.
    for column in ("Sector", "Industry"):
        value = lookup_krx_field(ticker, column)
        if value and re.search(r"바이오|제약|의약품", str(value)):
            return "바이오"
    return "일반"

if not DEMO_MODE:
    fundamentals_rows = []
    for stock in mention_summary["stock_name"]:
        stock_info = next((item for item in STOCK_DICTIONARY if item["name"] == stock), None)
        ticker = stock_info["ticker"] if stock_info else None
        corp_code = resolve_corp_code(stock, ticker)  # 3-2단계에서 정의됨 (우선주 → 보통주 법인으로 매핑)

        if not corp_code:
            print(f"  · {stock}: DART 법인 매핑 없음(코인/비상장 등) → 중립값으로 처리")
            fundamentals_rows.append({
                "stock_name": stock, "sector": "미확인", "market_cap_billion": None,
                "debt_ratio": 100.0, "is_administrative_issue": 0,
                "audit_opinion": "확인불가", "major_holder_change_pct": 0.0,
            })
            continue

        try:
            debt_ratio = fetch_debt_ratio(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 부채비율 조회 실패: {type(e).__name__} — {e}")
            debt_ratio = None
        try:
            audit_opinion = fetch_audit_opinion(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 감사의견 조회 실패: {type(e).__name__} — {e}")
            audit_opinion = None
        try:
            is_admin_issue = fetch_administrative_issue(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 관리종목 이력 조회 실패: {type(e).__name__} — {e}")
            is_admin_issue = 0
        try:
            holder_change = fetch_major_holder_change_pct(corp_code, dart_api_key)
        except Exception as e:
            print(f"  ✗ {stock} 최대주주 지분 변동 조회 실패: {type(e).__name__} — {e}")
            holder_change = 0.0

        market_cap = lookup_krx_field(ticker, "Marcap")
        fundamentals_rows.append({
            "stock_name": stock,
            "sector": guess_sector_label(ticker),
            "market_cap_billion": (market_cap / 1e8) if market_cap else None,  # 억원 단위로 환산
            "debt_ratio": debt_ratio if debt_ratio is not None else 100.0,  # 조회 실패 시 중립값
            "is_administrative_issue": is_admin_issue,
            "audit_opinion": audit_opinion if audit_opinion is not None else "확인불가",
            "major_holder_change_pct": holder_change,
        })
        time.sleep(0.3)  # DART 호출량 제한 완화

    fundamentals_df = pd.DataFrame(fundamentals_rows)
else:
    print("DEMO_MODE=True → 실제 DART/KRX 조회를 건너뛰고 샘플 재무 데이터를 사용합니다.")
    SAMPLE_FUNDAMENTALS = [
        {"stock_name": "현대약품", "sector": "바이오", "market_cap_billion": 180, "debt_ratio": 145.0,
         "is_administrative_issue": 0, "audit_opinion": "한정", "major_holder_change_pct": -8.5},
        {"stock_name": "SK하이닉스", "sector": "반도체", "market_cap_billion": 135000, "debt_ratio": 38.0,
         "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.1},
        {"stock_name": "삼성전자", "sector": "반도체", "market_cap_billion": 400000, "debt_ratio": 27.0,
         "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.0},
    ]
    fundamentals_df = pd.DataFrame(SAMPLE_FUNDAMENTALS)

# market_cap_billion 결측치(코인 등)는 극단적으로 취급되지 않도록 관측된 값의 중앙값으로 채움
if fundamentals_df["market_cap_billion"].notna().any():
    fundamentals_df["market_cap_billion"] = fundamentals_df["market_cap_billion"].fillna(
        fundamentals_df["market_cap_billion"].median()
    )
else:
    fundamentals_df["market_cap_billion"] = fundamentals_df["market_cap_billion"].fillna(1.0)

# 부채비율이 높을수록, 관리종목일수록, 감사의견이 '적정'이 아닐수록, 최대주주 지분이 급변할수록 불안정 → 감점
fundamentals_df["debt_risk"] = normalize01(fundamentals_df["debt_ratio"])
fundamentals_df["audit_risk"] = (fundamentals_df["audit_opinion"] != "적정").astype(float)
fundamentals_df["holder_change_risk"] = normalize01(fundamentals_df["major_holder_change_pct"].abs())

fundamentals_df["stability_score"] = 1 - (
    0.4 * fundamentals_df["debt_risk"]
    + 0.3 * fundamentals_df["is_administrative_issue"]
    + 0.2 * fundamentals_df["audit_risk"]
    + 0.1 * fundamentals_df["holder_change_risk"]
)

# 회의 반영: 소액으로도 주가가 흔들리는 "취약 기업" 특성 — 시가총액이 작을수록, 바이오 섹터일수록 소액 개입에 취약
fundamentals_df["small_cap_risk"] = normalize01(1 / fundamentals_df["market_cap_billion"])
fundamentals_df["sector_risk"] = (fundamentals_df["sector"] == "바이오").astype(float)
fundamentals_df["vulnerability_score"] = (
    0.6 * fundamentals_df["small_cap_risk"] + 0.4 * fundamentals_df["sector_risk"]
)

fundamentals_df[[
    "stock_name", "sector", "market_cap_billion", "debt_ratio", "is_administrative_issue",
    "audit_opinion", "stability_score", "vulnerability_score",
]]

---
# 6단계. 거래량 급등 신호 계산 + 종합 판단 (건전 투자 유도 북극성 지표)

"거래량이 평소 대비 300% 이상 급등"하는 현상 자체가 고전적인 이상치 탐지 문제입니다. `FinanceDataReader`로
KRX 실제 일별 시세를 받아와 20일 이동평균 대비 당일 거래량 배율을 계산하고, 아래 "종합 판단" 단계에서
설명 가능한 규칙의 입력으로 사용합니다.


In [ ]:
import numpy as np

VOLUME_LOOKBACK_DAYS = 40  # 20거래일 이동평균 계산에 필요한 여유(주말·공휴일 감안)

def fetch_real_volume_history(ticker, mention_date):
    import FinanceDataReader as fdr
    start = (mention_date - pd.Timedelta(days=VOLUME_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    end = pd.Timestamp.now().strftime("%Y-%m-%d")
    ohlcv = fdr.DataReader(ticker, start, end)
    if ohlcv is None or ohlcv.empty or "Close" not in ohlcv.columns or "Volume" not in ohlcv.columns:
        return None
    ohlcv = ohlcv.copy()
    ohlcv.index.name = "Date"
    ohlcv = ohlcv.reset_index()
    ohlcv["price_change_pct"] = ohlcv["Close"].pct_change() * 100
    ohlcv["day"] = range(len(ohlcv))
    return ohlcv[["day", "Volume", "price_change_pct"]].rename(columns={"Volume": "volume"})

if not DEMO_MODE:
    volume_frames = []
    for _, row in mention_summary.iterrows():
        stock = row["stock_name"]
        stock_info = next((item for item in STOCK_DICTIONARY if item["name"] == stock), None)
        ticker = stock_info["ticker"] if stock_info else None

        if not ticker:
            print(f"  · {stock}: 시세 데이터 없음(코인 등) → 거래량 급등 판단에서 제외")
            continue
        try:
            mention_date = pd.to_datetime(row["first_mentioned"])
            hist = fetch_real_volume_history(ticker, mention_date)
            if hist is None or hist.empty:
                print(f"  ✗ {stock}: 시세 데이터를 가져오지 못했습니다.")
                continue
            hist = hist.copy()
            hist["stock_name"] = stock
            volume_frames.append(hist)
        except Exception as e:
            print(f"  ✗ {stock} 시세 조회 실패: {type(e).__name__} — {e}")

    volume_df = (
        pd.concat(volume_frames, ignore_index=True) if volume_frames
        else pd.DataFrame(columns=["day", "volume", "price_change_pct", "stock_name"])
    )
else:
    print("DEMO_MODE=True → 실제 시세 조회를 건너뛰고 샘플 거래량/가격 데이터를 사용합니다.")
    # 데모용 샘플 일별 거래량/가격 데이터 (실제 수치 아님)
    np.random.seed(42)
    volume_rows = []
    for stock in fundamentals_df["stock_name"]:
        base_volume = np.random.randint(500_000, 2_000_000)
        for day in range(20):
            volume = int(base_volume * np.random.uniform(0.8, 1.2))
            price_change_pct = np.random.uniform(-2, 2)
            volume_rows.append({"stock_name": stock, "day": day, "volume": volume, "price_change_pct": price_change_pct})
        # 마지막 날, "현대약품"만 인위적으로 거래량 급등(리딩방 추천 직후 상황을 재현)
        if stock == "현대약품":
            volume_rows[-1]["volume"] = int(base_volume * 4.2)
            volume_rows[-1]["price_change_pct"] = 18.5
    volume_df = pd.DataFrame(volume_rows)

def compute_volume_ratio(group):
    group = group.sort_values("day").copy()
    rolling_avg = group["volume"].rolling(window=19, min_periods=5).mean().shift(1)
    group["volume_ratio"] = group["volume"] / rolling_avg
    return group

if len(volume_df):
    volume_df = pd.concat(
        [compute_volume_ratio(g) for _, g in volume_df.groupby("stock_name")],
        ignore_index=True,
    )
    volume_df["volume_spike_flag"] = volume_df["volume_ratio"] >= 3.0  # 300% 이상 급등 (참고용 표시, 최종 판정은 다음 셀의 규칙이 담당)
    latest_volume = volume_df.sort_values("day").groupby("stock_name").tail(1)
else:
    latest_volume = pd.DataFrame(columns=["stock_name", "day", "volume", "volume_ratio", "price_change_pct", "volume_spike_flag"])

latest_volume[["stock_name", "day", "volume", "volume_ratio", "price_change_pct", "volume_spike_flag"]]

## 종합 판단: 설명 가능한 규칙이 최종 결정, 통계적 이상치는 교차검증만

2~5단계에서 만든 피처(`pump_news_score`, `news_price_mismatch`, `stability_score`, `vulnerability_score`)와
6단계의 거래량 피처(`volume_ratio`)로 **최종 판단은 사람이 검증 가능한 규칙**이 내립니다. 비교할 종목이
충분히 쌓였을 때만 Isolation Forest로 "규칙이 놓쳤을 수 있는 통계적 이상치"를 추가로 훑어보되, 이 신호가
규칙의 최종 판단을 뒤집지는 않습니다.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

final_df = (
    mention_summary
    .merge(stock_news[["stock_name", "pump_news_score", "news_price_mismatch"]], on="stock_name", how="left")
    .merge(fundamentals_df[["stock_name", "sector", "stability_score", "vulnerability_score"]], on="stock_name", how="left")
    .merge(latest_volume[["stock_name", "volume_ratio", "price_change_pct", "volume_spike_flag"]], on="stock_name", how="left")
)

final_df["pump_news_score"] = final_df["pump_news_score"].fillna(0)
final_df["news_price_mismatch"] = final_df["news_price_mismatch"].fillna(False)
final_df["stability_score"] = final_df["stability_score"].fillna(0.5)
final_df["vulnerability_score"] = final_df["vulnerability_score"].fillna(0.0)
final_df["volume_spike_flag"] = final_df["volume_spike_flag"].fillna(False)

# ── 최종 판단: 설명 가능한 규칙 (투자자 보호 도구는 "왜 경고했는지" 사람이 검증할 수 있어야 함) ──
# 회의 반영: 소액 개입에 취약한(소형주·바이오) 기업일수록, 더 낮은 pump_news_score에서도 경고가 뜨도록
# 임계값 자체를 낮춥니다 (취약도 1.0이면 임계값 0.5 → 0.3까지 낮아짐).
final_df["pump_threshold"] = 0.5 - 0.2 * final_df["vulnerability_score"]

def classify(row):
    if row["volume_spike_flag"] and row["pump_news_score"] >= row["pump_threshold"] and row["stability_score"] < 0.5:
        return "⚠ 투자 주의 (묻지마 추종매수 위험)"
    if row["volume_spike_flag"] and row["stability_score"] >= 0.5 and row["pump_news_score"] < row["pump_threshold"]:
        return "정상 호재 가능성 (모니터링)"
    return "특이사항 없음"

final_df["investment_guidance"] = final_df.apply(classify, axis=1)

# ── 보조 교차검증: Isolation Forest — 규칙이 놓쳤을 수 있는 통계적 이상 패턴을 추가로만 훑어봅니다.
#     최종 결정을 뒤집지 않으며, 비교 종목이 너무 적으면(5개 미만) 통계적으로 무의미해 아예 건너뜁니다.
FEATURE_COLS = [
    "pump_news_score", "news_price_mismatch", "stability_score",
    "vulnerability_score", "volume_ratio", "price_change_pct",
]

if len(final_df) >= 5:
    feature_df = final_df[FEATURE_COLS].copy()
    feature_df["news_price_mismatch"] = feature_df["news_price_mismatch"].astype(float)
    feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

    X_scaled = StandardScaler().fit_transform(feature_df.values)
    contamination = min(0.3, max(0.05, 3 / len(final_df)))
    iso = IsolationForest(n_estimators=300, contamination=contamination, random_state=42)
    iso.fit(X_scaled)
    final_df["stat_outlier_score"] = -iso.score_samples(X_scaled)
    final_df["stat_outlier_flag"] = iso.predict(X_scaled) == -1
else:
    final_df["stat_outlier_score"] = None
    final_df["stat_outlier_flag"] = False
    print(f"비교 대상 종목이 {len(final_df)}개뿐이라 통계적 교차검증(Isolation Forest)은 신뢰도가 낮아 건너뜁니다. "
          f"(종목이 5개 이상 쌓이면 자동으로 활성화됩니다)")

final_df[[
    "stock_name", "sector", "mention_count", "pump_news_score", "news_price_mismatch",
    "stability_score", "vulnerability_score", "pump_threshold",
    "volume_ratio", "volume_spike_flag", "investment_guidance",
    "stat_outlier_score", "stat_outlier_flag",
]]

### 북극성 지표 계산 방식 (건전 투자 유도율)

> **북극성 지표**: 거래량 급등이 발생한 종목 중, 사전에 "투자 주의" 경고가 노출된 비율.
>
> `건전 투자 유도율 = (급등 + 주의 라벨이 붙은 종목 수) / (전체 거래량 급등 종목 수) × 100`
>
> 실제 서비스에서는 여기에 "경고 노출 이후 실제 매수 억제 효과"(예: 경고 노출 전후 거래량 변화)까지 연결해야
> 진짜 북극성이 완성되지만, 이 노트북에서는 종목 자료만으로 계산 가능한 부분까지만 구현했습니다.


In [ ]:
spiked = final_df[final_df["volume_spike_flag"]]
if len(spiked):
    healthy_guidance_rate = (spiked["investment_guidance"] == "⚠ 투자 주의 (묻지마 추종매수 위험)").mean() * 100
    print(f"거래량 급등 종목 {len(spiked)}개 중 '투자 주의' 경고 비율(건전 투자 유도율): {healthy_guidance_rate:.1f}%")
else:
    print("이번 데이터에서는 거래량 급등(300%+)이 감지된 종목이 없습니다.")

---
# 7단계. 결과 저장 및 다운로드


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(final_df["stock_name"], final_df["pump_news_score"], label="pump_news_score")
plt.bar(final_df["stock_name"], final_df["stability_score"], alpha=0.6, label="stability_score")
plt.title("종목별 뉴스 의심도 vs 기업 안정성 점수")
plt.ylabel("score")
plt.legend()
plt.tight_layout()
plt.show()

OUT_CSV_PATH = "investment_guidance_result.csv"
OUT_XLSX_PATH = "investment_guidance_result.xlsx"
final_df.to_csv(OUT_CSV_PATH, index=False, encoding="utf-8-sig")
final_df.to_excel(OUT_XLSX_PATH, index=False)
print(f"결과 저장 완료 → {OUT_CSV_PATH}, {OUT_XLSX_PATH}")

try:
    from google.colab import files
    files.download(OUT_CSV_PATH)
    files.download(OUT_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


---
## 한계 및 유의사항

### 1~2단계: 수집·종목 추출
- `DEMO_MODE=True`일 때의 샘플 데이터(메시지·뉴스·재무·거래량)는 전부 파이프라인 동작 확인용 예시이며 실제 수치가 아닙니다.
- 종목 추출은 KRX 전체 상장 종목 사전(FinanceDataReader)으로 확장했지만 여전히 substring 매칭이라, 일반 단어와
  우연히 겹치는 짧은 사명은 오탐 가능성이 남아 있습니다. 종목 수가 많아지면 메시지당 매칭 비교 비용도 늘어나
  대량 크롤링에서는 속도 저하가 있을 수 있습니다(느려지면 Aho-Corasick 등 다중 패턴 매칭으로 교체 검토).
  근본적으로는 한국어 개체명 인식(NER) 모델로 교체해야 더 정확해집니다.
- KRX 종목 리스트는 실행할 때마다 새로 받아오므로, 실행 시점에 따라 상장폐지·신규상장이 반영되어 사전 내용이
  달라질 수 있습니다.

### 3단계: 뉴스·언론사·공시
- 네이버 뉴스 검색 API는 호출량 제한이 있고 언론사 정보를 명시적으로 주지 않아 URL 도메인으로 추정합니다.
- 언론사 신뢰도는 KPF "참여 언론사 현황"(131개)을 1차 화이트리스트로, sinmundeul.com에서 확인한 도메인으로
  보강했습니다. 연합뉴스·JTBC 등 KPF 명단에는 없지만 잘 알려진 매체는 `OTHER_MAJOR_MEDIA`로 별도 등록해
  중간 신뢰도(0.85)를 부여합니다. 다만 (1) 일부 KPF 참여매체(주로 지역주간지)는 도메인 매핑이 안 돼 있어
  "명단 외"로 잘못 분류될 수 있고, (2) `OTHER_MAJOR_MEDIA` 목록도 공식 화이트리스트가 아니라 제가 임의로
  추린 것이므로, 계속 검증·확장이 필요합니다.
- 공시 매칭은 이제 실제 Open DART API를 씁니다. 다만 `corpCode.xml`은 매 실행마다 전체 상장사 목록(수만 건)을
  새로 받아오므로 다소 느릴 수 있고, 코인·비상장 종목은 DART 매핑이 아예 없어 항상 "공시 없음"으로 처리됩니다
  — 이건 데이터 누락이지 실제 의심 신호가 아니므로 해석 시 주의가 필요합니다.
- **우선주(대덕1우, 삼성SDI우 등)는 DART에 별도 법인이 없고 보통주 발행 법인과 동일한 법인**이라, 우선주
  티커로는 DART corp_code가 절대 매핑되지 않습니다. 종목명에서 우선주 표기("우", "1우", "2우B" 등)를 떼어
  보통주 이름으로 재조회하도록 처리했지만, 이 이름 패턴 가정이 100% 보장되진 않으니 매핑 결과를 한 번씩
  확인하는 걸 권장합니다.

### 4단계: 뉴스 버스트 판별
- "작전세력 의심 보도" 판별은 근접중복·버스트 타이밍 등 통계적 의심 신호일 뿐입니다. 실제 허위·과장 보도인지는
  기사 원문 검증과 사람의 판단이 반드시 필요하며, 이 결과만으로 특정 매체·기사를 단정해서는 안 됩니다.
- 첨부 가이드라인의 채널 유형(A/B/C/D) 프로파일링을 추가했지만, 아직 참고용 설명 자료일 뿐 최종
  `investment_guidance` 판정에는 반영되지 않았습니다.

### 5단계: 기업 건전성
- 이제 실제 Open DART API(재무제표·감사의견·최대주주 변동)와 KRX 리스트(시가총액)를 씁니다. 다만 일부 DART
  세부 API는 보고서 종류·연도에 따라 데이터가 없을 수 있어, 실패 시 중립값(부채비율 100%, 감사의견
  "확인불가")으로 대체합니다 — 조회 실패와 실제 위험 신호를 혼동하지 않도록 주의하세요.
- 업종(바이오 여부)은 KRX 리스팅 텍스트에서 키워드로 근사 추정한 것이라 부정확할 수 있어, 실전에서는 정식
  업종분류 코드로 교체해야 합니다.
- 관리종목 지정 여부는 전용 DART API가 없어 공시 제목에서 "관리종목" 문구를 검색하는 방식으로 근사했습니다
  — 실전에서는 KRX 상장공시시스템(KIND)과 직접 대조하는 게 더 정확합니다.

### 6단계: 거래량·최종 판정
- 거래량 급등은 이제 FinanceDataReader로 실제 KRX 일별 시세를 가져와 계산합니다(20일 이동평균 대비 배율).
  다만 상장폐지·거래정지·최근 상장 종목은 이력이 부족해 이동평균 계산이 불안정할 수 있습니다.
- 최종 판정은 사람이 검증 가능한 규칙이 내리고, Isolation Forest는 비교 종목이 5개 이상일 때만 보조
  교차검증 신호(`stat_outlier_flag`)로 참고합니다 — 통계적 이상치는 "달라서 이상한 것"이지 "위험해서 이상한
  것"이 아니고, 표본이 적으면 `contamination` 파라미터가 사실상 결과를 정하는 것과 다름없어 최종 결정에서
  제외했습니다.
- 규칙 임계값(0.5, 3.0배 등)은 회의 논의를 반영한 초기 설계일 뿐 검증된 값이 아니므로, 실제 사례가 쌓이면
  재조정이 필요합니다. `stat_outlier_flag`가 규칙과 다르게 나오는 종목은 규칙의 사각지대일 수 있어 사람이
  별도 검토하는 걸 권장합니다.
- 가이드라인 11장의 "반드시 확인할 반례"(대형주 동시 언급, 정책 발표 후 테마 전체 상승 등)를 SK하이닉스·
  삼성전자 데모 사례로 일부 반영했지만, 실전에서는 별도 회귀 테스트 세트로 오탐률을 주기적으로 점검해야 합니다.

### 북극성 지표
- "건전 투자 유도율"은 이 노트북에서 계산 가능한 부분(경고 노출 비율)까지만 구현했습니다. 실제 "묻지마
  매수 억제 효과"까지 검증하려면 경고 노출 전후의 실제 거래 데이터가 추가로 필요합니다.
